<a href="https://colab.research.google.com/github/esalinasbio/taller-modelado-biomolecular/blob/master/notebooks/04_Docking_RNA_rDock.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Parte 2b — Docking sobre RNA con **rDock**

**Taller de Modelado Biomolecular**

---

En el ejercicio anterior acoplamos un ligando a una proteína con AutoDock Vina y
funcionó bien. Después probamos el **mismo programa** sobre un RNA y funcionó peor.

La explicación que dimos fue que la función de puntuación de Vina se calibró con
complejos proteína-ligando, y que los sitios de unión en RNA son distintos en casi
todo lo que esa función mide: más cargados, más polares, dominados por apilamiento
entre bases, a menudo dependientes de Mg²⁺.

Ahora la pregunta obvia: **¿existe una herramienta hecha para esto?**

Sí. **rDock** se desarrolló y validó específicamente para blancos de ácidos nucleicos.
Vamos a acoplar exactamente el mismo sistema y comparar.

---

In [ ]:
#@title 1 - Instalar conda
#@markdown Se va a reiniciar la sesión, no pasa nada
!pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
#@title 2  Instalar rDock y el restode dependencias
import os, time

_t0 = time.time()

!mamba install -q -y -c bioconda rdock openbabel
!pip install py3Dmol rdkit

print(f"Listo en {time.time() - _t0:.1f} s")

import os, subprocess, numpy as np
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, rdMolAlign, Draw
RDLogger.DisableLog("rdApp.*")
import py3Dmol, matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "font.size": 10})
C_REC, C_XTAL, C_DOCK, GRIS = "#2E8B8B", "#E8813A", "#7c3aed", "#64748b"

print("\n=== Comprobación de binarios ===")
ok = True
for b in ("rbcavity", "rbdock", "obabel"):
    r = subprocess.run(["which", b], capture_output=True, text=True)
    print(f"  {b:10s} {r.stdout.strip() or 'NO ENCONTRADO'}")
    ok &= bool(r.stdout.strip())
print("RBT_ROOT =", os.environ.get("RBT_ROOT", "(no definido)"))
print(f"\n{'Todo listo' if ok else '*** FALTA ALGO ***'} "
      f"({time.time()-_t0:.0f} s)")

---
## El sistema: un *riboswitch* de adenina

`1Y26` es el aptámero de un riboswitch que une **adenina**. Los riboswitches son
elementos en el mRNA que cambian de conformación al unir un metabolito pequeño y con
eso regulan la expresión del gen que sigue. Son RNA que hacen de sensor químico, sin
proteína de por medio.

Volvemos a hacer **redocking**: quitamos la adenina de la estructura cristalográfica,
la acoplamos de vuelta y medimos qué tan lejos quedó de la posición real.


In [ ]:
#@title 3 - Descargar y separar
PDB_ID      = "1Y26"   #@param {type:"string"}
COD_LIGANDO = "ADE"    #@param {type:"string"}

import urllib.request
urllib.request.urlretrieve(f"https://files.rcsb.org/download/{PDB_ID}.pdb", "entrada.pdb")

rec, lig = [], []
for l in open("entrada.pdb"):
    if l.startswith("ENDMDL"): break
    if not l.startswith(("ATOM", "HETATM")): continue
    resn = l[17:20].strip()
    if resn == COD_LIGANDO: lig.append(l)
    elif resn in ("HOH", "WAT"): continue
    elif l.startswith("ATOM"): rec.append(l)
open("receptor.pdb", "w").write("".join(rec) + "END\n")
open("ligando_xtal.pdb", "w").write("".join(lig) + "END\n")

# reconstruir el ligando con ordenes de enlace correctos
urllib.request.urlretrieve(
    f"https://files.rcsb.org/ligands/download/{COD_LIGANDO}_ideal.sdf", "ideal.sdf")
plantilla = Chem.MolFromMolFile("ideal.sdf")
crudo = Chem.MolFromPDBBlock(open("ligando_xtal.pdb").read(), removeHs=True)
xtal = AllChem.AssignBondOrdersFromTemplate(plantilla, crudo)
flat = AllChem.AssignBondOrdersFromTemplate(plantilla, crudo)
smiles = Chem.MolToSmiles(plantilla)

# referencia en formato SD, que es lo que rDock espera
Chem.MolToMolFile(xtal, "ref_ligando.sd")

print("="*56)
print(f"  Átomos del receptor (RNA) : {len(rec)}")
print(f"  Átomos del ligando        : {len(lig)}")
print(f"  SMILES                    : {smiles}")
print(f"  Enlaces rotables          : "
      f"{Chem.rdMolDescriptors.CalcNumRotatableBonds(xtal)}")
print("="*56)
print('''
Fíjense en el número de enlaces rotables: la adenina es prácticamente rígida.
Desde el punto de vista de la BÚSQUEDA, este problema es mucho más fácil que el
indinavir del cuaderno anterior.

Si aun así el resultado con Vina fue peor, el problema no era la búsqueda.
Era la PUNTUACIÓN.
''')

AllChem.Compute2DCoords(flat)

print("Ligando:")
Draw.MolToImage(flat, size=(300, 220))

In [ ]:
#@title 4 - Ver el sitio de unión
v = py3Dmol.view(width=780, height=450)
v.addModel(open("receptor.pdb").read(), "pdb")
v.setStyle({}, {"cartoon": {"color": C_REC}})
v.addStyle({}, {"stick": {"radius": 0.10, "color": C_REC, "opacity": 0.6}})
v.addModel(Chem.MolToMolBlock(xtal), "mol")
v.setStyle({"model": 1}, {"stick": {"radius": 0.3, "colorscheme": "orangeCarbon"}})
v.zoomTo({"model": 1}); v.show()
print("La adenina queda enterrada en el núcleo del pliegue del RNA, no en la superficie.")
print("Eso es lo que da la especificidad del riboswitch.")

---
## Preparar para rDock

rDock trabaja distinto a Vina y conviene ver en qué:

- El receptor va en **MOL2**, no PDBQT.
- La caja de búsqueda no se define con coordenadas y tamaño, sino con un
  **mapeo de cavidad**: `rbcavity` detecta la cavidad real del receptor. Nosotros le
  damos el ligando cristalográfico como referencia para saber cuál cavidad nos
  interesa (`RbtLigandSiteMapper`).
- Todo se configura en un archivo de parámetros `.prm` en texto plano.

Ese mapeo de cavidad es conceptualmente distinto de la caja de Vina: en vez de una
región rectangular arbitraria, rDock construye el **volumen accesible** dentro del
sitio. Para un pocket profundo y irregular es una descripción
mucho mejor.

In [ ]:
#@title 5 - Preparar receptor y cavidad
RADIO_CAVIDAD  = 6.0   #@param {type:"number"}
VOLUMEN_MINIMO = 100   #@param {type:"integer"}

!obabel receptor.pdb -O receptor.mol2
print("receptor.mol2:", os.path.getsize("receptor.mol2"), "bytes")

prm = f'''RBT_PARAMETER_FILE_V1.00
TITLE riboswitch_{PDB_ID}

RECEPTOR_FILE receptor.mol2
RECEPTOR_FLEX 3.0

SECTION MAPPER
    SITE_MAPPER RbtLigandSiteMapper
    REF_MOL ref_ligando.sd
    RADIUS {RADIO_CAVIDAD}
    SMALL_SPHERE 1.0
    MIN_VOLUME {VOLUMEN_MINIMO}
    MAX_CAVITIES 1
    VOL_INCR 0.0
    GRIDSTEP 0.5
END_SECTION

SECTION CAVITY
    SCORING_FUNCTION RbtCavityGridSF
    WEIGHT 1.0
END_SECTION
'''
open("sistema.prm", "w").write(prm)
print(prm)

r = subprocess.run(["rbcavity", "-was", "-d", "-r", "sistema.prm"],
                   capture_output=True, text=True)
print(r.stdout[-1200:])
if r.returncode != 0:
    print("STDERR:", r.stderr[-800:])
else:
    print("\nArchivos de cavidad generados:",
          [f for f in os.listdir('.') if f.endswith(('.as', '.grd'))])

Fíjense en `RECEPTOR_FLEX 3.0`: rDock permite que las **cadenas laterales terminales
del receptor** roten durante el acoplamiento.

Es una flexibilidad muy limitada —nada que ver con lo que vieron en dinámica
molecular— pero es más de lo que Vina hacía, que trataba al receptor como
completamente rígido.

> ### Pregunta
> Después de la Parte 1, donde vieron el RNA moverse varios ángstroms:
> **¿les parece suficiente?**

In [ ]:
#@title 6 - Acoplar
N_CORRIDAS = 100   #@param {type:"integer"}
SEMILLA    = 42   #@param {type:"integer"}

# confórmero nuevo desde SMILES: no partimos de la respuesta
m = Chem.MolFromSmiles(smiles)
AllChem.EmbedMolecule(m, randomSeed=SEMILLA)
AllChem.MMFFOptimizeMolecule(m)
with Chem.SDWriter("ligando.sd") as writer:
    writer.write(m)
rmsd_ini = rdMolAlign.CalcRMS(Chem.RemoveHs(m), Chem.RemoveHs(xtal))
print(f"RMSD del confórmero inicial vs. cristal: {rmsd_ini:.1f} Å  "
      f"(prueba de que no partimos de la respuesta)\n")

t0 = time.time()
r = subprocess.run(["rbdock", "-i", "ligando.sd", "-o", "acoplado",
                    "-r", "sistema.prm", "-p", "dock.prm",
                    "-n", str(N_CORRIDAS), "-s", str(SEMILLA)],
                   capture_output=True, text=True)
print(f"rbdock terminó en {time.time()-t0:.0f} s (rc={r.returncode})")
if r.returncode != 0:
    print(r.stdout[-1500:]); print("STDERR:", r.stderr[-1000:])
else:
    print("Salida:", [f for f in os.listdir('.') if f.startswith('acoplado')])

In [ ]:
#@title 7 - Evaluar: ¿qué tan cerca quedó del cristal?
sup = Chem.SDMolSupplier("acoplado.sd", removeHs=False)
poses = [x for x in sup if x is not None]
print(f"Poses generadas: {len(poses)}")

xn = Chem.RemoveHs(xtal)
filas = []
for i, p in enumerate(poses):
    try:
        score = float(p.GetProp("SCORE"))
        inter = float(p.GetProp("SCORE.INTER")) if p.HasProp("SCORE.INTER") else np.nan
    except Exception:
        score, inter = np.nan, np.nan
    rms = rdMolAlign.CalcRMS(Chem.RemoveHs(p), xn)
    filas.append((i+1, score, inter, rms))

filas.sort(key=lambda x: x[1])   # ordenar por puntuación
print(f"\n{'pose':>5}{'SCORE':>10}{'INTER':>10}{'RMSD (Å)':>11}")
print("-"*38)
for n, s, it, rm in filas[:10]:
    marca = "  <-- acierto" if rm < 2.0 else ""
    print(f"{n:>5}{s:>10.2f}{it:>10.2f}{rm:>11.2f}{marca}")

rms_all = np.array([f[3] for f in filas])
sc_all  = np.array([f[1] for f in filas])
mejor_rdock = rms_all.min()
rmsd_top    = filas[0][3]
print(f"\nRMSD de la pose mejor puntuada : {rmsd_top:.2f} Å")
print(f"Mejor RMSD entre todas          : {mejor_rdock:.2f} Å")
print(f"Poses por debajo de 2 Å         : {(rms_all<2).sum()} de {len(rms_all)}")

In [ ]:
#@title 8 · Ver el resultado y comparar con Vina
RMSD_VINA = None   #@param {type:"number"}

mejor = poses[filas[0][0]-1]
v = py3Dmol.view(width=780, height=460)
v.addModel(open("receptor.pdb").read(), "pdb")
v.setStyle({}, {"cartoon": {"color": "#cbd5e1", "opacity": 0.5}})
v.addStyle({}, {"stick": {"radius": 0.08, "color": "#cbd5e1", "opacity": 0.5}})
v.addModel(Chem.MolToMolBlock(xtal), "mol")
v.setStyle({"model": 1}, {"stick": {"radius": 0.28, "colorscheme": "orangeCarbon"}})
v.addModel(Chem.MolToMolBlock(mejor), "mol")
v.setStyle({"model": 2}, {"stick": {"radius": 0.20, "colorscheme": "purpleCarbon"}})
v.zoomTo({"model": 1}); v.show()
print("Naranja = cristal (la verdad) · Morado = mejor pose de rDock")

fig, ax = plt.subplots(1, 2, figsize=(11.5, 3.8))
ax[0].scatter(rms_all, sc_all, s=45,
              c=["#16a34a" if r < 2 else C_DOCK for r in rms_all],
              edgecolors="k", linewidths=0.4)
ax[0].axvline(2.0, ls="--", c="#b91c1c", lw=1.3)
ax[0].set_xlabel("RMSD respecto al cristal (Å)")
ax[0].set_ylabel("SCORE de rDock")
ax[0].set_title("¿La puntuación distingue las poses buenas?")
ax[0].grid(alpha=0.25)

etiquetas, valores, colores = ["rDock"], [rmsd_top], [C_DOCK]
if RMSD_VINA > 0:
    etiquetas.append("Vina"); valores.append(RMSD_VINA); colores.append(GRIS)
ax[1].bar(etiquetas, valores, color=colores, alpha=0.9)
ax[1].axhline(2.0, ls="--", c="#b91c1c", lw=1.3)
ax[1].set_ylabel("RMSD de la pose mejor puntuada (Å)")
ax[1].set_title("Herramienta general vs. herramienta específica")
ax[1].grid(alpha=0.25, axis="y")
plt.tight_layout(); plt.show()
